# Phase 6c — Contrastive LoRA (leakage-free text alignment)
Fixes Phase 5's text-branch failures by training the way CLIP actually works: pull each image embedding toward its own class's text embedding, push it away from the other 7 -- never concatenate the label's text into the classifier's input.

**At inference, the model only sees the image.** It predicts by checking which of the 8 text embeddings the image embedding is now closest to (same mechanic as Phase 5b's zero-shot, but now the image encoder is LoRA-fine-tuned to actually respond to these descriptions). No class label ever enters the model as a feature -- this cannot leak.

**Improvements are toggles, not all-on-by-default.** Run baseline first, then enable ONE toggle at a time and compare against the previous run before stacking more -- isolates what actually helps.

```python
USE_AUGMENTATION = False   # toggle 1
USE_FOCAL_LOSS   = False   # toggle 2
N_EPOCHS         = 10      # toggle 3 -- raise once baseline is confirmed working
```

In [ ]:
%pip install -q setuptools packaging ftfy regex tqdm peft scikit-learn
%pip install -q -U torchao
%pip install -q git+https://github.com/openai/CLIP.git
!git clone --depth 1 https://github.com/Michi-3000/EyeCLIP.git || true

import pathlib
clip_path = pathlib.Path("EyeCLIP/eyeclip/clip.py")
clip_text = clip_path.read_text()
clip_text = clip_text.replace("from pkg_resources import packaging", "import packaging")
clip_path.write_text(clip_text)

import sys
sys.path.insert(0, "EyeCLIP")

import torch, eyeclip
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from peft import LoraConfig, get_peft_model

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

CKPT_PATH = "/kaggle/input/eyeclip-checkpoint/eyeclip_visual.pt"          # CHANGE to actual path
IMAGE_ROOT = Path("/kaggle/input/datasets/faruq05/rop-vl/ROP-VL/Image")
SPLIT_DIR = Path("/kaggle/input/datasets/faruq05/rop-vl/ROP-VL/split")
DISEASE_DESC_PATH = "/kaggle/input/datasets/faruq05/rop-vl/disease_description.xlsx"  # CHANGE if path differs

# ===== TOGGLES -- change ONE at a time between runs =====
USE_AUGMENTATION = False
USE_FOCAL_LOSS   = False
N_EPOCHS         = 10
FOCAL_GAMMA      = 2.0   # only used if USE_FOCAL_LOSS=True
RUN_NAME         = "baseline"  # CHANGE this each run so output files don't overwrite each other

In [ ]:
model, preprocess = eyeclip.load("ViT-B/32", device=DEVICE, jit=False)
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
state = ckpt.get("model_state_dict") or ckpt.get("state_dict") or ckpt
model.load_state_dict(state, strict=False)
model.eval()
print("EyeCLIP loaded")

TARGET_MODULES = ["out_proj", "c_fc", "c_proj"]
lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=TARGET_MODULES, lora_dropout=0.05, bias="none")
lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()
assert sum(p.numel() for p in lora_model.parameters() if p.requires_grad) > 0, "LoRA attached to 0 params"

## Load splits, encode class text descriptions (frozen text tower, never fine-tuned)

In [ ]:
train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df   = pd.read_csv(SPLIT_DIR / "val.csv")
test_df  = pd.read_csv(SPLIT_DIR / "test.csv")

classes_sorted = sorted(train_df["categories"].unique())
class_to_idx = {c: i for i, c in enumerate(classes_sorted)}
NUM_CLASSES = len(classes_sorted)

CATEGORY_TO_FOLDER = {
    "normal": "Normal", "Stage 1 ROP": "Stage 1 ROP", "Stage 2 ROP": "Stage 2 ROP",
    "Stage 3 ROP": "Stage 3 ROP", "Stage 4 ROP": "Stage 4 ROP", "Stage 5 ROP": "Stage 5 ROP",
    "A-ROP": "A-ROP", "Laser-treated ROP": "Laser-treated ROP",
}
assert not (set(classes_sorted) - set(CATEGORY_TO_FOLDER.keys())), "unmapped class -- check folder names"

desc_df = pd.read_excel(DISEASE_DESC_PATH)
assert hasattr(eyeclip, "tokenize"), "eyeclip module has no tokenize()"
tokens = eyeclip.tokenize(desc_df["description"].tolist(), truncate=True).to(DEVICE)
with torch.no_grad():
    text_embeddings_raw = model.encode_text(tokens).float()   # [8, 512], text tower stays frozen throughout
text_class_order = desc_df["disease"].tolist()
assert not (set(classes_sorted) - set(text_class_order)), "class missing a text description"
# reorder text embeddings to match classes_sorted so index i always means classes_sorted[i]
reorder_idx = [text_class_order.index(c) for c in classes_sorted]
text_embeddings = text_embeddings_raw[reorder_idx]  # [NUM_CLASSES, 512], frozen constant during training
text_embeddings_norm = F.normalize(text_embeddings, dim=1)

## Dataset with optional augmentation (toggle)
Standard fundus-image augmentation: mild rotation, brightness/contrast jitter, horizontal flip. Kept mild -- aggressive augmentation can distort clinically meaningful features (ridge shape, vessel patterns).

In [ ]:
aug_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
])

class ROPContrastiveDataset(Dataset):
    """Row -> (preprocessed image tensor, class idx). No text/metadata concatenated -- leakage-free by construction."""
    def __init__(self, df, image_root, preprocess, class_to_idx, augment=False):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.preprocess = preprocess
        self.class_to_idx = class_to_idx
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        folder = CATEGORY_TO_FOLDER[row["categories"]]
        img = Image.open(self.image_root / folder / row["image"]).convert("RGB")
        if self.augment:
            img = aug_transform(img)   # applied on PIL image, before EyeCLIP's own preprocess/resize/normalize
        x = self.preprocess(img)
        y = self.class_to_idx[row["categories"]]
        return x, y

train_ds = ROPContrastiveDataset(train_df, IMAGE_ROOT, preprocess, class_to_idx, augment=USE_AUGMENTATION)
val_ds   = ROPContrastiveDataset(val_df,   IMAGE_ROOT, preprocess, class_to_idx, augment=False)  # never augment val/test
test_ds  = ROPContrastiveDataset(test_df,  IMAGE_ROOT, preprocess, class_to_idx, augment=False)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)

## Contrastive loss + optional focal weighting (toggle)
**Contrastive step:** normalize image embedding, compute cosine similarity to all 8 (frozen) text embeddings, treat this as an 8-way classification problem where the "logits" are similarities -- cross-entropy on that IS the CLIP-style contrastive objective, no text ever enters as an input feature, only as the fixed comparison target.
**Focal loss:** down-weights easy/majority-class examples so the model focuses more on rare classes (Stage 4/5) than plain class-weighting alone does.

In [ ]:
class FocalLoss(nn.Module):
    """Standard focal loss: (1-p_t)^gamma * cross_entropy. Reduces to weighted CE when gamma=0."""
    def __init__(self, class_weights, gamma=2.0):
        super().__init__()
        self.class_weights = class_weights
        self.gamma = gamma
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.class_weights, reduction="none")
        p_t = torch.exp(-ce)
        return ((1 - p_t) ** self.gamma * ce).mean()

class_counts = train_df["categories"].map(class_to_idx).value_counts().reindex(range(NUM_CLASSES), fill_value=1).values
class_weights = torch.tensor(1.0 / np.maximum(class_counts, 1), dtype=torch.float32, device=DEVICE)

if USE_FOCAL_LOSS:
    criterion = FocalLoss(class_weights, gamma=FOCAL_GAMMA)
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights)

LOGIT_SCALE = 100.0  # standard CLIP temperature scaling on cosine similarity before cross-entropy

def contrastive_logits(img_emb):
    """img_emb [B,512] -> logits [B,NUM_CLASSES] via cosine similarity to frozen text embeddings."""
    img_norm = F.normalize(img_emb, dim=1)
    return LOGIT_SCALE * (img_norm @ text_embeddings_norm.T)

## Train LoRA with contrastive loss
Only LoRA's injected parameters are trainable -- text tower and rest of EyeCLIP stay frozen. Model selection on val macro-F1, never on test.

In [ ]:
optimizer = torch.optim.Adam([p for p in lora_model.parameters() if p.requires_grad], lr=1e-4)

best_val_f1, best_lora_state = -1.0, None
history = []

for epoch in range(N_EPOCHS):
    lora_model.train()
    total_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        img_emb = lora_model.encode_image(x).float()
        logits = contrastive_logits(img_emb)
        loss = criterion(logits, y)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * x.size(0)

    lora_model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            img_emb = lora_model.encode_image(x).float()
            preds = contrastive_logits(img_emb).argmax(dim=1).cpu().numpy()
            val_preds.extend(preds); val_labels.extend(y.numpy())
    _, _, val_f1, _ = precision_recall_fscore_support(val_labels, val_preds, average=None, zero_division=0)
    val_macro_f1 = val_f1.mean()
    history.append({"epoch": epoch, "train_loss": total_loss / len(train_ds), "val_macro_f1": val_macro_f1})
    print(f"epoch {epoch}  loss={total_loss/len(train_ds):.4f}  val_macro_f1={val_macro_f1:.4f}")

    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        best_lora_state = {k: v.clone() for k, v in lora_model.state_dict().items() if "lora_" in k}

lora_model.load_state_dict(best_lora_state, strict=False)
print(f"\nbest val macro-F1: {best_val_f1:.4f}")
pd.DataFrame(history).to_csv(f"/kaggle/working/phase6c_{RUN_NAME}_history.csv", index=False)

## Test evaluation -- image-to-text similarity only, no label ever used as input

In [ ]:
lora_model.eval()
test_preds, test_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        img_emb = lora_model.encode_image(x).float()
        preds = contrastive_logits(img_emb).argmax(dim=1).cpu().numpy()
        test_preds.extend(preds); test_labels.extend(y.numpy())

acc = accuracy_score(test_labels, test_preds)
precision, recall, f1_per_class, support = precision_recall_fscore_support(
    test_labels, test_preds, labels=range(NUM_CLASSES), average=None, zero_division=0)
macro_f1 = f1_per_class.mean()

print(f"[{RUN_NAME}] Contrastive LoRA -- Test accuracy: {acc:.4f}   Macro-F1: {macro_f1:.4f}\n")
print("Reference -- Phase 5b (frozen zero-shot, collapsed to majority class): acc=0.3031  macro_f1=0.0582")
print("Reference -- Phase 6 (LoRA image-only, classifier head): acc=0.2375  macro_f1=0.1659")
print("Reference -- LoRA+metadata (Phase 6b, no text): acc=0.3219  macro_f1=0.2060\n")

for c, p, r, f, s in zip(classes_sorted, precision, recall, f1_per_class, support):
    print(f"  {c:20s}  P={p:.3f}  R={r:.3f}  F1={f:.3f}  n={s}")

pd.DataFrame([{
    "run_name": RUN_NAME, "use_augmentation": USE_AUGMENTATION, "use_focal_loss": USE_FOCAL_LOSS,
    "n_epochs": N_EPOCHS, "accuracy": acc, "macro_f1": macro_f1, "best_val_f1": best_val_f1,
}]).to_csv(f"/kaggle/working/phase6c_{RUN_NAME}_result.csv", index=False)

## How to use this notebook for the one-by-one comparison
1. Run as-is first (`RUN_NAME="baseline"`, all toggles False, `N_EPOCHS=10`). Record accuracy/macro_f1.
2. Change ONLY `USE_AUGMENTATION = True`, `RUN_NAME = "augmentation"`, rerun from the toggles cell down. Compare to baseline.
3. Reset augmentation, change ONLY `USE_FOCAL_LOSS = True`, `RUN_NAME = "focal_loss"`, rerun. Compare to baseline.
4. Reset focal loss, raise `N_EPOCHS` (e.g. 25), `RUN_NAME = "more_epochs"`, rerun -- watch `val_macro_f1` in the printed history for overfitting (val plateauing or dropping while train loss keeps falling).
5. Only stack toggles together once you know which ones individually helped -- each `phase6c_<run_name>_result.csv` lets you build one final comparison table across all runs.

**Why this design avoids leakage:** the class label never appears as a concatenated input feature anywhere in this notebook -- text embeddings are only ever used as a frozen comparison target for cosine similarity, computed identically at train and test time. This is structurally the same mechanism as Phase 5b's zero-shot test, so it cannot reproduce Phase 5a's shortcut.